In [ ]:
from _common import *

## Notebook 03 -- Ridge

Rung 1 (primary): nested Optuna tuning across all 79 session-folds, one model, under
the fairness contract in `design/models/model-comparison-protocol.md`. Writes
`fold_metrics_ridge.json` for notebook 06's comparison table. Own MLflow db
(`mlflow_ridge.db`) -- see `docs/logs/20260826_224433_plan_notebook03_split.md`.

In [ ]:
import mlflow

MLFLOW_DB_PATH = ROOT_FOLDER / 'notebooks' / 'mlflow_ridge.db'
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("03_ridge")

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import json

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Provisional, carried from the 9-fold pilot -- confirm after timing fold 0 on the
# real 79-fold / 8-participant data. See plan doc SS4 (scale warning).
MAX_TRIALS = 100   # M -- hard cap on trials per fold
PATIENCE = 15     # N -- stop early if the best value hasn't6 improved in this many trials

In [ ]:
def make_model(params):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=params['alpha'])),
    ])


def make_objective(fold):
    def objective(trial):
        # Widened 1e3 -> 1e6 (2026-08-28): every fold was landing on alpha=993.78,
        # right at the old upper edge -- the search never found a true interior
        # optimum. Matches the range design/models/model-comparison-protocol.md
        # always specified ("nine orders of magnitude") but the code never
        # implemented. strain_x is an exact linear combination of dx columns
        # (rank-deficient design matrix), so a much larger alpha than the sklearn
        # default range is expected to be needed -- see protocol doc SS1.
        alpha = trial.suggest_float('alpha', 1e-3, 1e6, log=True)
        return inner_cv_score(fold, {'alpha': alpha}, make_model)
    return objective


def stop_when_flat(study, trial):
    # study-level patience -- the only stopping rule (Optuna pruning is OFF, see
    # plan doc traps). Stop once PATIENCE trials have passed with no new best.
    if trial.number - study.best_trial.number >= PATIENCE:
        study.stop()

### Rung 1 -- nested tuning, all 79 folds

For each fold: tune `alpha` with Optuna against `inner_cv_score` (the 3-session inner
rotation from the manifest), refit on the full 8 train sessions with the best `alpha`,
score on the held-out test session with `evaluate()`, log to MLflow.

**Resumable.** This run takes hours -- if you interrupt it (Ctrl+C, closing the
kernel, going home), re-running this cell picks up where it left off. It loads
`fold_metrics_ridge.json` if present, skips any (participant, test_session) already
in it, and saves after every new fold, not just at the end.

In [ ]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


def _save_checkpoint(results):
    with open(FOLD_METRICS_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


FOLD_METRICS_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_ridge.json'

if FOLD_METRICS_PATH.exists():
    with open(FOLD_METRICS_PATH) as f:
        fold_results = json.load(f)
else:
    fold_results = []

done_keys = {(r['participant_id'], r['test_session']) for r in fold_results}
print(f"resuming with {len(fold_results)} folds already done, {len(SESSION_FOLDS) - len(fold_results)} remaining")

In [ ]:
study_first_new = None   # the first fold actually run THIS session -- used for the
                          # convergence diagnostic below, not for scoring

for fold in SESSION_FOLDS:
    p_num = fold['participant_id']
    test_session = fold['test_session']

    if (p_num, test_session) in done_keys:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    if study_first_new is None:
        study_first_new = study

    best_params = {'alpha': study.best_params['alpha']}

    train_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'].isin(fold['train_sessions']))
    test_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'] == test_session)

    # leak guard -- the outer test session must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'session_id'] == test_session).any()
    assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(fold['train_sessions'])

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"p{p_num}_s{test_session}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results.append({
        'participant_id': p_num,
        'fold': fold['fold'],
        'test_session': test_session,
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys.add((p_num, test_session))
    _save_checkpoint(fold_results)

    print(p_num, test_session, len(study.trials), result['overall'])

print(f"done -- {len(fold_results)} / {len(SESSION_FOLDS)} folds in fold_metrics_ridge.json")

### Convergence check -- one fold only

Evidence the patience budget did not stop the search too early. One fold (the first
one actually run this session), not all 79 -- 79 plots would be noise, and this is a
sanity check, not a per-fold report. Skips silently if this run resumed with nothing
new to do.

In [ ]:
import sys, plotly
print(sys.executable)
print(plotly.__file__)

In [ ]:
from optuna.visualization import plot_optimization_history

# One-off: fresh Optuna study on fold 0, purely for the convergence plot --
# doesn't touch fold_metrics_ridge.json or done_keys, since all 79 folds are
# already checkpointed and this isn't meant to re-save anything.

sampler = TPESampler(seed=RANDOM_SEED)
plot_study = optuna.create_study(direction='minimize', sampler=sampler)
plot_study.optimize(make_objective(SESSION_FOLDS[0]), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

plot_optimization_history(plot_study)

### Rung 2 -- fine-tuned cross-mount (pending)

Refit with a slice of the held-out session, test on the remainder. **Slice size and
position not yet decided** -- plan doc SS6 open item, carried from the pilot. Do not
guess this; confirm with Guillaume before writing it.

In [ ]:
# TODO(rung 2): pending slice-size decision. Placeholder -- not run.

### Rung 3 -- cross-subject, cold

Held-out unit is a whole **participant**, not a session -- every session they have is
excluded from training, and there is no fine-tuning (that is rung 4). Uses
`SUBJECT_FOLDS` (8 folds, one per participant) and `INNER_VAL_PARTICIPANTS`
(`_common.py` SS2b) -- a fixed 3-participant round-robin, the subject-level analog of
rung 1's inner-3 session rotation. Tuning goes through `inner_cv_score_subject()`
(`_common.py` SS6b) instead of `inner_cv_score()`.

Same `alpha` search space, same `MAX_TRIALS`/`PATIENCE`, same `make_model()` as rung 1
-- only the fold source and the inner-CV function differ. Writes
`fold_metrics_ridge_rung3.json` (separate checkpoint file, keyed by `participant_id`
not `(participant_id, test_session)`). Logs to the same `mlflow_ridge.db` /
`03_ridge` experiment as rung 1, with a distinct run-name scheme (`rung3_p{p}` vs
rung 1's `p{p}_s{s}`) so the two rungs' runs never collide.

**Per decision 3 (chat log 2026-08-28): time fold 0 before committing to a budget.**
Rung 3's training set per fold is ~7-8x rung 1's (7 whole participants' full data vs
~8 sessions of 1 participant) -- do not assume `MAX_TRIALS=100`/`PATIENCE=15` survives
unchanged, the same caution that turned out to matter for both Ridge and XGBoost at
rung 1 scale (plan doc SS4a/SS4c).

In [ ]:
def make_objective_subject(fold):
    def objective(trial):
        # Widened 1e3 -> 1e6 (2026-08-28), matching the same fix in rung 1's
        # make_objective() -- see that cell's comment. All 8 rung-3 folds had
        # landed on alpha=993.78, and every fold hit exactly 47 trials, which is
        # the signature of a monotonic "more regularization is always better"
        # landscape that never found a true interior optimum inside the old range.
        alpha = trial.suggest_float('alpha', 1e-3, 1e6, log=True)
        return inner_cv_score_subject(fold, INNER_VAL_PARTICIPANTS[fold['fold']], {'alpha': alpha}, make_model)
    return objective

# make_model() and stop_when_flat() are reused unchanged from rung 1 (cell above).

#### Fold-0 timing check -- confirm the budget before the full 8-fold run

5 trials, no patience callback -- same smoke-test shape used for rung 1 (plan doc
SS4a) and for XGBoost's rung-1 check (SS4c). Not a real tuning run; purely a
per-trial wall-clock measurement to extrapolate from before spending real compute on
all 8 folds.

In [ ]:
import time

_t0 = time.time()
timing_sampler = TPESampler(seed=RANDOM_SEED)
timing_study = optuna.create_study(direction='minimize', sampler=timing_sampler)
timing_study.optimize(make_objective_subject(SUBJECT_FOLDS[0]), n_trials=5)
_t1 = time.time()

per_trial = (_t1 - _t0) / 5
projected_fold_minutes = per_trial * MAX_TRIALS / 60
projected_total_hours = per_trial * MAX_TRIALS * len(SUBJECT_FOLDS) / 3600

print(f"rung 3 fold 0, 5 trials, no patience: {per_trial:.2f} s/trial")
print(f"worst case (no patience firing) at MAX_TRIALS={MAX_TRIALS}: "
      f"{projected_fold_minutes:.1f} min/fold, {projected_total_hours:.1f} h for all {len(SUBJECT_FOLDS)} folds")

#### Rung 3 -- nested tuning, all 8 folds

**Review the timing check above before running this cell.** If the projected total
does not survive (same failure mode as XGBoost's rung 1, plan doc SS4c), lower
`MAX_TRIALS`/`PATIENCE` for rung 3 specifically rather than assuming rung 1's numbers
transfer -- do not silently shrink the search space to compensate without flagging it.

**Resumable**, same pattern as rung 1: loads `fold_metrics_ridge_rung3.json` if
present, skips any `participant_id` already in it, saves after every new fold.

In [ ]:
FOLD_METRICS_RUNG3_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_ridge_rung3.json'

if FOLD_METRICS_RUNG3_PATH.exists():
    with open(FOLD_METRICS_RUNG3_PATH) as f:
        fold_results_rung3 = json.load(f)
else:
    fold_results_rung3 = []

done_keys_rung3 = {r['participant_id'] for r in fold_results_rung3}
print(f"resuming with {len(fold_results_rung3)} folds already done, {len(SUBJECT_FOLDS) - len(fold_results_rung3)} remaining")


def _save_checkpoint_rung3(results):
    with open(FOLD_METRICS_RUNG3_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


for fold in SUBJECT_FOLDS:
    test_p = int(fold['test_participant'].removeprefix('participant'))

    if test_p in done_keys_rung3:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective_subject(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    best_params = {'alpha': study.best_params['alpha']}

    train_participant_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
    train_mask = FEATURES['participant_id'].isin(train_participant_nums)
    test_mask = FEATURES['participant_id'] == test_p

    # leak guard -- the outer test participant must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'participant_id'] == test_p).any()
    assert set(FEATURES.loc[train_mask, 'participant_id'].unique()) == set(train_participant_nums)

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"rung3_p{test_p}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results_rung3.append({
        'participant_id': test_p,
        'fold': fold['fold'],
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys_rung3.add(test_p)
    _save_checkpoint_rung3(fold_results_rung3)

    print(test_p, len(study.trials), result['overall'])

print(f"done -- {len(fold_results_rung3)} / {len(SUBJECT_FOLDS)} folds in fold_metrics_ridge_rung3.json")

In [ ]:
rmses_rung3 = np.array([r['metrics']['overall']['rmse'] for r in fold_results_rung3])

print(f"Ridge, rung 3, {len(rmses_rung3)} folds")
print(f"mean RMSE {rmses_rung3.mean():.2f} %MVC (SD {rmses_rung3.std(ddof=1):.2f}, "
      f"range {rmses_rung3.min():.2f}-{rmses_rung3.max():.2f})")

# Flat per-rung summary, matching rung 1's style above -- superseded once the
# EchoForce-style per-model table (decision 5, chat log 2026-08-28) exists.

### Summary

Aggregate across all 79 folds -- mean, SD, range. Do not report this number alone;
the by-phase and by-level breakdown in `fold_metrics_ridge.json` is the fuller story
(protocol SS4 -- pre-cue rows are ~1% MVC and drag the aggregate down).

In [ ]:
rmses = [r['metrics']['overall']['rmse'] for r in fold_results]
rmses = np.array(rmses)

print(f"Ridge, rung 1, {len(rmses)} folds")
print(f"mean RMSE {rmses.mean():.2f} %MVC (SD {rmses.std(ddof=1):.2f}, range {rmses.min():.2f}-{rmses.max():.2f})")